In [ ]:
#read the parquet file
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.optimize import brentq




In [ ]:
df = pd.read_parquet("unpivoted_bid_offer_train_clean.parquet", engine="pyarrow")

required = [
        "option_bid", "option_ask",
        "underlying_settlement", "strike",
        "expiration", "date", "option_type"
    ]

df = df.dropna(subset=required)
df.head()

df['date'] = pd.to_datetime(df['date'])

In [ ]:

fixing_dates = pd.bdate_range(
    start="2021-02-01",
    end="2021-02-28"
)


valuation_date = pd.Timestamp("2020-01-06")

theta = (
    (fixing_dates - valuation_date).days / 365
).to_numpy()


In [ ]:
print(theta)

In [ ]:
futures_df = (
    df[[
        "date",
        "future_contract_code",
        "underlying_settlement",
        "underlying_bid",
        "underlying_ask"
    ]]
    .drop_duplicates(subset=["date", "future_contract_code"])
    .copy()
)
options_df = df.dropna(
    subset=[
        "option_bid",
        "option_ask",
        "strike",
        "underlying_settlement",
        "expiration"
    ]
).copy()


options_df["mid_price"] = (
    options_df["option_bid"] + options_df["option_ask"]
) / 2


options_df["settlement_minus_mid"] = (
    options_df["settlement_price"] -
    options_df["mid_price"]
)

print(
    options_df[
        ["settlement_price", "mid_price",
         "settlement_minus_mid"]
    ].describe()
)

In [ ]:
futures_df.head()

In [ ]:
N = 20
n1, n2 = 18, 2
w1, w2 = n1/N, n2/N



In [ ]:
def calculate_d1_d2(f, K, sigma, T,t):
    d1 = (np.log(f / K) + 0.5*sigma ** 2 * (T-t)) / (sigma * np.sqrt(T-t))
    d2 = d1 - sigma * np.sqrt(T-t)
    return d1,d2


In [ ]:
def black_76(F, K, sigma, t, T, option_type='call'):
    d1, d2 = calculate_d1_d2(F, K, sigma,T, t)

    if option_type == 'call':
        return (F * norm.cdf(d1) - K * norm.cdf(d2))

    elif option_type == 'put':
        return K * norm.cdf(-d2) - F * norm.cdf(-d1)

    else:
        raise ValueError("Invalid option type")


def implied_volatility(f, K, T, t, market_price, option_type='call', tol=1e-5, max_iterations=100):
    objective_function = lambda sigma: black_76(f, K, sigma, t, T, option_type) - market_price
    return brentq(objective_function, 1e-6, 5, xtol=tol, maxiter=max_iterations)


F = 75
K = 78
sigma = 0.30
t = 0
T = 1

call = black_76(F, K, sigma, t, T, "call")
put = black_76(F, K, sigma, t, T, "put")

assert np.isclose(call - put, F - K)

estimated_sigma = implied_volatility(
    F, K, T, t, call, "call"
)

assert np.isclose(estimated_sigma, sigma)

In [ ]:


F = 75
K = 78
tau = 1.0
market_price = 7.5

iv = implied_volatility(
    f=F,
    K=K,
    t=0,
    T=tau,
    market_price=market_price,
    option_type="call"
)

reconstructed_price = black_76(
    F, K, iv, 0, tau, "call"
)

print(f"Market price: {market_price:.6f}")
print(f"Model price:  {reconstructed_price:.6f}")
print(f"Difference:   {reconstructed_price - market_price:.8f}")

In [ ]:
D = 0.5


def simulate_correlated_futures(
    F1, F2,
    sigma1, sigma2,
    rho, theta, n1,D,
    n_simulations=10000
):
    N = len(theta)
    K = 78
    sigma = np.where(
        np.arange(N) < n1,
        sigma1,
        sigma2
    )

    mu = -0.5 * sigma**2 * theta
    min_theta = np.minimum.outer(theta, theta)

    Sigma = np.zeros((N, N))

    for a in range(N):
        for b in range(N):
            #minimum of theta[a] and theta[b]
            min_theta = min(theta[a], theta[b])
            if a < n1 and b < n1:
            # Both fixings use Apr21
                Sigma[a, b] = sigma1**2 * min_theta
            elif a >= n1 and b >= n1:
            # Both fixings use May21
                Sigma[a, b] = sigma2**2 * min_theta
            else:
            # One fixing uses Apr21, the other May21
                Sigma[a, b] = rho * sigma1 * sigma2 * min_theta

    Chol = np.linalg.cholesky(Sigma)
    Z = np.random.normal(size=(N, n_simulations))
    eta = mu[:, np.newaxis] + Chol @ Z
    forward_prices = np.exp(eta)
    A1 = np.mean(forward_prices[:n1, :], axis=0)
    A2 = np.mean(forward_prices[n1:, :], axis=0 )
    A = w1 * A1 + w2 * A2
    Payoff_call = np.maximum(A - K, 0)
    Payoff_put = np.maximum(K - A, 0)
    price_call = D*np.mean(Payoff_call)
    price_put = D*np.mean(Payoff_put)
    std_call = D*np.std(Payoff_call)/np.sqrt(n_simulations)
    std_put = D*np.std(Payoff_put)/np.sqrt(n_simulations)
    return price_call, price_put, std_call, std_put


In [ ]:
def price_asian_basket():

    K1 = f1 * K / Favg
    K2 = f2 * K / Favg

    sigma1 = float(smile_apr21(K1))
    sigma2 = float(smile_may21(K2))

    varA1 =  
    return None


"""
class forward():
    def __init__(self, K, tau):
        self.K = K
        self.tau = tau
        self.type = "forward"
    
    def price(self, F, sigma, t, T):
        return F * np.exp(-0.01 * (T-t)) - self.K * np.exp(-0.01 * (T-t))

class asian_basket():
    def __init__(self, K, tau, n1, n2):
        self.K = K
        self.tau = tau
        self.n1 = n1
        self.n2 = n2
        self.type = "asian_basket"
    
    def price(self, F1, F2, sigma1, sigma2, rho, theta, D):
        return simulate_correlated_futures(
            F1, F2,
            sigma1, sigma2,
            rho, theta,
            self.n1,D
        )[:2]

    K1 = 


In [ ]:
daily_options = options_df[
    options_df["date"] == "2020-01-06"
].copy()

print(
    daily_options[
        ["future_contract_code", "option_contract_code"]
    ].drop_duplicates()
)

In [ ]:
valuation_date = pd.Timestamp("2020-01-06")

apr21_options = daily_options[
    daily_options["future_contract_code"] == "COJ21"
].copy()

may21_options = daily_options[
    daily_options["future_contract_code"] == "COK21"
].copy()

print("Apr21 observations:", len(apr21_options))
print("May21 observations:", len(may21_options))

In [ ]:
print(
    apr21_options["underlying_settlement"].unique()
)

print(
    may21_options["underlying_settlement"].unique()
)

F1 = 61.4
F2 = 61.05
Favg = w1*F1+w2*F2

In [ ]:
columns = [
    "strike",
    "option_type",
    "expiration",
    "time_to_expiry_days",
    "option_bid",
    "option_ask",
    "settlement_price"
]

print("APR21 OPTIONS")
print(apr21_options[columns].sort_values("strike").to_string(index=False))

print("\nMAY21 OPTIONS")
print(may21_options[columns].sort_values("strike").to_string(index=False))

In [ ]:
def calculate_smile_points(options):
    data = options.copy()

    # Select out-of-the-money options
    is_otm = (
        ((data["option_type"].str.lower() == "put") &
         (data["strike"] < data["underlying_settlement"])) |
        ((data["option_type"].str.lower() == "call") &
         (data["strike"] >= data["underlying_settlement"]))
    )

    data = data.loc[is_otm].copy()

    # Calculate mid-prices
    data["mid_price"] = (
        data["option_bid"] + data["option_ask"]
    ) / 2

    # Time to expiration in years
    data["tau"] = data["time_to_expiry_days"] / 365

    # Compute implied volatility
    def compute_iv(row):
        try:
            return implied_volatility(
                f=row["underlying_settlement"],
                K=row["strike"],
                T=row["tau"],
                t=0,
                market_price=row["mid_price"],
                option_type=row["option_type"].lower()
            )
        except (ValueError, RuntimeError):
            return np.nan

    data["implied_vol"] = data.apply(compute_iv, axis=1)

    return data.dropna(subset=["implied_vol"])

In [ ]:
apr21_smile_data = calculate_smile_points(apr21_options)
may21_smile_data = calculate_smile_points(may21_options)

print(
    apr21_smile_data[
        ["strike", "implied_vol"]
    ].sort_values("strike")
)

print(
    may21_smile_data[
        ["strike", "implied_vol"]
    ].sort_values("strike")
)

In [ ]:
#plot the implied volatility smiles for both contracts

plt.figure(figsize=(10, 6))
plt.plot(apr21_smile_data["strike"], apr21_smile_data["implied_vol"], "o-",label="APR21")
plt.plot(may21_smile_data["strike"], may21_smile_data["implied_vol"], "o-", label="MAY21")
plt.xlabel("Strike")
plt.ylabel("Implied Volatility")
plt.title("Volatility Smiles")
plt.legend()
plt.show()

In [ ]:
def select_otm(df):
    F = df["underlying_settlement"]

    return df[
        ((df["option_type"].str.lower() == "put") &
         (df["strike"] < F)) |
        ((df["option_type"].str.lower() == "call") &
         (df["strike"] >= F))
    ].copy()

apr_otm = select_otm(apr21_smile_data).sort_values("strike")
may_otm = select_otm(may21_smile_data).sort_values("strike")

plt.figure(figsize=(10, 5))

plt.plot(apr_otm["strike"], apr_otm["implied_vol"],
         "o-", label="APR21")

plt.plot(may_otm["strike"], may_otm["implied_vol"],
         "o-", label="MAY21")

plt.xlabel("Strike")
plt.ylabel("Implied volatility")
plt.title("OTM Volatility Smiles")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
from scipy.interpolate import PchipInterpolator

smile_apr21 = PchipInterpolator(
    apr_otm["strike"],
    apr_otm["implied_vol"],
    extrapolate=False
)

smile_may21 = PchipInterpolator(
    may_otm["strike"],
    may_otm["implied_vol"],
    extrapolate=False
)

In [ ]:
K = 78
def get_volatility(F1,F2,Favg,K, smile_apr21, smile_may21):
    K1 = K/Favg*F1
    K2 = K/Favg*F2

    sigma_apr = float(smile_apr21(K1))
    sigma_may = float(smile_may21(K2))
    return sigma_apr, sigma_may

get_volatility(F1,F2,Favg,K, smile_apr21, smile_may21)

print(f"Apr21 volatility: {sigma_apr:.4%}")
print(f"May21 volatility: {sigma_may:.4%}")

In [ ]:
prices = futures_df.pivot(
    index="date",
    columns="future_contract_code",
    values="underlying_settlement"
)

prices = prices[["COJ21", "COK21"]].dropna()

returns = np.log(prices / prices.shift(1)).dropna()

rho = returns["COJ21"].corr(returns["COK21"])

print(f"Estimated correlation: {rho:.4f}")